# FTIR 성분 분석 (Spectral Matching)

DPT 파일로 측정한 미지 시료를 RIST FTIR 라이브러리와 비교하여 성분을 동정합니다.

**분석 흐름**
1. 미지 시료 DPT 로드 → 전처리 (스무딩, 베이스라인 보정)
2. 라이브러리(217개 스펙트럼) 로드 → 공통 파수 그리드로 보간
3. 코사인 유사도(Cosine Similarity)로 상위 매칭 소재 추출
4. 미지 시료 vs 상위 매칭 스펙트럼 비교 시각화

## 0. 라이브러리 임포트

In [6]:
import os
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from sklearn.metrics.pairwise import cosine_similarity
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("패키지 로드 완료")

패키지 로드 완료


## 1. 분석 설정
분석할 DPT 파일 경로와 파라미터를 설정합니다.

In [7]:
# ── 분석 대상 파일 ───────────────────────────────────────────────────
SAMPLE_FILE  = "sample_TR(transmittance).0.dpt"   # 미지 시료 DPT 파일
SAMPLE_LABEL = "Unknown Sample"                    # 그래프에 표시할 이름

# ── 라이브러리 경로 ──────────────────────────────────────────────────
LIBRARY_DIR  = "data/RIST_FTIR_Library"
MANIFEST     = os.path.join(LIBRARY_DIR, "manifest.csv")

# ── 전처리 파라미터 ──────────────────────────────────────────────────
WN_MIN       = 650     # 분석 파수 하한 (cm⁻¹)
WN_MAX       = 4000    # 분석 파수 상한 (cm⁻¹)
N_GRID       = 1750    # 공통 그리드 포인트 수 (WN_MAX-WN_MIN 간격)
SMOOTH       = True    # Savitzky-Golay 스무딩 적용 여부
SMOOTH_WIN   = 11      # 스무딩 윈도우 (홀수)
SMOOTH_POLY  = 3       # 다항식 차수

# ── 결과 출력 ───────────────────────────────────────────────────────
TOP_N        = 10      # 상위 매칭 결과 개수
PLOT_TOP_N   = 5       # 비교 그래프에 표시할 상위 개수

## 2. 전처리 함수 정의
스펙트럼 로드, 공통 그리드 보간, 베이스라인 보정, 정규화 함수를 정의합니다.

In [9]:
# 공통 파수 그리드
GRID = np.linspace(WN_MIN, WN_MAX, N_GRID)


def load_dpt(path):
    """Bruker DPT (wavenumber, intensity) 파일 로드"""
    df = pd.read_csv(path, header=None, names=["wn", "y"])
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    df = df[(df["wn"] >= WN_MIN) & (df["wn"] <= WN_MAX)]
    return df.sort_values("wn")


def load_csv(path):
    """라이브러리 CSV (wavenumber, absorbance) 로드"""
    df = pd.read_csv(path, comment="#")
    df.columns = [c.strip() for c in df.columns]
    wn_col = next(c for c in df.columns if "wave" in c.lower() or "wn" in c.lower())
    ab_col = next(c for c in df.columns if "abs" in c.lower() or "int" in c.lower())
    df = df[[wn_col, ab_col]].rename(columns={wn_col: "wn", ab_col: "y"})
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    df = df[(df["wn"] >= WN_MIN) & (df["wn"] <= WN_MAX)]
    return df.sort_values("wn")


def baseline_als(y, lam=1e5, p=0.01, n_iter=10):
    """Asymmetric Least Squares 베이스라인 보정"""
    from scipy.sparse import diags, csc_matrix
    from scipy.sparse.linalg import spsolve
    L = len(y)
    ones = np.ones(L)
    # 2차 차분 행렬 (L-2, L) sparse — format 인수는 단따옴표 사용
    D = diags([ones, -2 * ones, ones], [0, 1, 2], shape=(L - 2, L), format='csc')
    H = lam * D.T.dot(D)   # (L, L) sparse penalty matrix
    w = np.ones(L)
    for _ in range(n_iter):
        W = diags(w, 0, shape=(L, L), format='csc')
        Z = spsolve(W + H, w * y)
        w = p * (y > Z) + (1 - p) * (y <= Z)
    return y - Z


def preprocess(wn, y):
    """보간 → 스무딩 → 베이스라인 보정 → Min-Max 정규화"""
    # 보간
    f = interp1d(wn, y, kind="linear", bounds_error=False, fill_value=0)
    y_grid = f(GRID)

    # 스무딩 (Savitzky-Golay)
    if SMOOTH:
        y_grid = savgol_filter(y_grid, window_length=SMOOTH_WIN, polyorder=SMOOTH_POLY)

    # 베이스라인 보정
    y_grid = baseline_als(y_grid)

    # Min-Max 정규화
    mn, mx = y_grid.min(), y_grid.max()
    if mx - mn > 1e-10:
        y_grid = (y_grid - mn) / (mx - mn)

    return y_grid


print("전처리 함수 정의 완료")

전처리 함수 정의 완료


## 3. 미지 시료 로드 및 전처리

In [10]:
raw = load_dpt(SAMPLE_FILE)
sample_vec = preprocess(raw["wn"].values, raw["y"].values)

# 원시 스펙트럼 + 전처리 결과 비교 플롯
fig_pre = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["원시 스펙트럼 (Raw)", "전처리 후 (정규화)"],
                        vertical_spacing=0.12)

fig_pre.add_trace(go.Scatter(x=raw["wn"], y=raw["y"],
                             mode="lines", line=dict(color="#555", width=1),
                             name="Raw"), row=1, col=1)
fig_pre.add_trace(go.Scatter(x=GRID, y=sample_vec,
                             mode="lines", line=dict(color="#2563eb", width=1.5),
                             name="Preprocessed"), row=2, col=1)

fig_pre.update_layout(
    title=f"전처리 확인 — {SAMPLE_LABEL}",
    height=500, plot_bgcolor="white", paper_bgcolor="#fafafa",
    showlegend=False,
)
fig_pre.update_xaxes(range=[WN_MAX, WN_MIN], title_text="Wavenumber (cm⁻¹)", row=2, col=1)
fig_pre.update_yaxes(showgrid=True, gridcolor="#e8e8e8")
fig_pre.show()

print(f"로드 완료: {len(raw)} 포인트 → 그리드 {N_GRID} 포인트")

로드 완료: 1736 포인트 → 그리드 1750 포인트


## 3-1. FTIR 분석 원리 — 피크와 작용기

FTIR은 화학 결합이 특정 파수(cm⁻¹)의 적외선을 흡수하는 원리를 이용합니다.  
**흡광도가 극대(피크)가 되는 파수 위치**를 보고 어떤 작용기가 있는지 동정합니다.

| 파수 범위 (cm⁻¹) | 작용기 |
|---|---|
| 3600 – 3200 | O-H stretch (수산기, 수분) |
| 3300 – 3100 | N-H stretch (아민, 아미드) |
| 3000 – 2850 | C-H stretch (알킬기) |
| 2260 – 2100 | C≡N / C≡C (니트릴, 알킨) |
| 1760 – 1690 | C=O stretch (카르보닐, 에스터, 케톤) |
| 1680 – 1620 | C=C / N-H bend (알켄, 아미드 II) |
| 1470 – 1350 | C-H bend (메틸, 메틸렌) |
| 1300 – 1000 | C-O / C-N stretch (지문 영역) |
| 900 – 700 | C-H oop bend (방향족, 비닐) |

> 피크의 **위치** + **모양(날카로운/넓은)** + **상대 강도** 조합으로 성분을 특정합니다.

## 3-2. 피크 검출 및 작용기 주석

In [22]:
from scipy.signal import find_peaks, peak_widths

# ══════════════════════════════════════════════════════════════════════
# 피크 검출 파라미터
# ══════════════════════════════════════════════════════════════════════
PEAK_HEIGHT     = 0.05   # 최소 정규화 흡광도
PEAK_PROMINENCE = 0.03   # 피크 돌출 정도
PEAK_DISTANCE   = 15     # 피크 간 최소 그리드 포인트 수

# ══════════════════════════════════════════════════════════════════════
# 작용기 규칙 — 외부 CSV에서 로드
# 파일: data/func_groups.csv  (// 로 시작하는 줄은 주석)
# 컬럼: center_wn | tolerance | name | color | note
# ★ tolerance가 좁을수록 높은 우선순위 (더 구체적)
# ══════════════════════════════════════════════════════════════════════
FUNC_GROUPS_FILE = "data/func_groups.csv"

_fg = pd.read_csv(FUNC_GROUPS_FILE, comment='/')
FUNC_GROUPS = [
    (int(r["center_wn"]), int(r["tolerance"]),
     str(r["name"]), str(r["color"]),
     "" if pd.isna(r["note"]) else str(r["note"]))
    for _, r in _fg.iterrows()
]
print(f"작용기 규칙 로드: {len(FUNC_GROUPS)}개  ←  {FUNC_GROUPS_FILE}")


# ══════════════════════════════════════════════════════════════════════
# assign_group: 중심파수 기준, 좁은 허용범위(구체적) 우선
# ══════════════════════════════════════════════════════════════════════
def assign_group(wn):
    candidates = [(tol, name, color, note)
                  for center, tol, name, color, note in FUNC_GROUPS
                  if abs(wn - center) <= tol]
    if not candidates:
        return "unknown", "#9ca3af", ""
    candidates.sort(key=lambda x: x[0])
    _, name, color, note = candidates[0]
    return name, color, note


# ══════════════════════════════════════════════════════════════════════
# detect_patterns: 복수 피크 조합으로 작용기 확인
# ══════════════════════════════════════════════════════════════════════
def has_peak_near(wns, center, tol=40):
    return any(abs(w - center) <= tol for w in wns)

def get_peak_near(wns, vals, center, tol=40):
    """center 근처 최대 강도 피크값 반환 (없으면 0)"""
    matches = [(v, w) for w, v in zip(wns, vals) if abs(w - center) <= tol]
    return max(matches)[0] if matches else 0.0

def detect_patterns(wns, vals):
    """
    복수 피크 조합으로 성분 패턴을 확인합니다.
    반환: [(확신도, 패턴명, 근거 설명)] 리스트 (확신도 내림차순)
    """
    results = []

    # ── 에스터 (Ester) ─────────────────────────────────────────────
    co = get_peak_near(wns, vals, 1735, 30)
    coc = get_peak_near(wns, vals, 1240, 35)
    if co > 0.1 and coc > 0.05:
        conf = min(1.0, (co + coc) / 1.5)
        results.append((conf, "에스터 (Ester)",
                        f"C=O ~1735 ({co:.2f}) + C-O-C ~1240 ({coc:.2f})"))

    # ── 카르복실산 (Carboxylic Acid) ────────────────────────────────
    co_acid = get_peak_near(wns, vals, 1720, 25)
    oh_broad = any(abs(w - 2800) <= 400 for w in wns)
    if co_acid > 0.1 and oh_broad:
        conf = min(1.0, co_acid * 1.2)
        results.append((conf, "카르복실산 (Carboxylic Acid)",
                        f"C=O ~1720 ({co_acid:.2f}) + broad O-H 2500-3300"))

    # ── 아미드 (Amide) ──────────────────────────────────────────────
    amide1 = get_peak_near(wns, vals, 1655, 25)
    amide2 = get_peak_near(wns, vals, 1550, 40)
    nh     = get_peak_near(wns, vals, 3300, 80)
    if amide1 > 0.08:
        conf = min(1.0, amide1 + amide2 * 0.5 + nh * 0.3)
        results.append((conf, "아미드 (Amide)",
                        f"아미드 I ~1655 ({amide1:.2f}) + 아미드 II ~1550 ({amide2:.2f})"))

    # ── 알코올 (Alcohol) ────────────────────────────────────────────
    oh = get_peak_near(wns, vals, 3400, 200)
    co_alc = max(get_peak_near(wns, vals, 1050, 35),
                 get_peak_near(wns, vals, 1100, 35))
    if oh > 0.1 and co_alc > 0.05:
        conf = min(1.0, (oh + co_alc) / 1.5)
        results.append((conf, "알코올 (Alcohol)",
                        f"O-H ~3400 ({oh:.2f}) + C-O ~1050-1100 ({co_alc:.2f})"))

    # ── 방향족 (Aromatic) ───────────────────────────────────────────
    ar_peaks = [get_peak_near(wns, vals, wn, 20) for wn in [1600, 1580, 1500]]
    ar_count = sum(1 for v in ar_peaks if v > 0.05)
    oop = max(get_peak_near(wns, vals, 760, 30),
              get_peak_near(wns, vals, 700, 30),
              get_peak_near(wns, vals, 840, 30))
    if ar_count >= 2 or (ar_count >= 1 and oop > 0.05):
        conf = min(1.0, sum(ar_peaks) * 0.5 + oop * 0.3)
        results.append((conf, "방향족 고리 (Aromatic)",
                        f"C=C ring {ar_count}개 피크 + C-H oop ({oop:.2f})"))

    # ── 플루오로폴리머 PVDF/PTFE ────────────────────────────────────
    cf = get_peak_near(wns, vals, 1170, 30)
    if cf > 0.1:
        conf = min(1.0, cf * 1.5)
        results.append((conf, "플루오로폴리머 (PVDF/PTFE)",
                        f"C-F ~1170 ({cf:.2f})"))

    # ── 실리콘 (Silicone) ───────────────────────────────────────────
    siosi = get_peak_near(wns, vals, 1100, 40)
    ch3si = get_peak_near(wns, vals, 1260, 30)
    if siosi > 0.15 and ch3si > 0.05:
        conf = min(1.0, (siosi + ch3si) / 1.5)
        results.append((conf, "실리콘 (Silicone/Silicate)",
                        f"Si-O-Si ~1100 ({siosi:.2f}) + Si-CH3 ~1260 ({ch3si:.2f})"))

    # ── 니트릴 (Nitrile) ────────────────────────────────────────────
    cn = get_peak_near(wns, vals, 2230, 25)
    if cn > 0.08:
        conf = min(1.0, cn * 2.0)
        results.append((conf, "니트릴 (Nitrile)", f"C≡N ~2230 ({cn:.2f}) 날카로운 피크"))

    # ── 알데히드 (Aldehyde) ─────────────────────────────────────────
    co_ald = get_peak_near(wns, vals, 1695, 18)
    ch_ald = get_peak_near(wns, vals, 2720, 30)
    if co_ald > 0.08 and ch_ald > 0.03:
        conf = min(1.0, (co_ald + ch_ald))
        results.append((conf, "알데히드 (Aldehyde)",
                        f"C=O ~1695 ({co_ald:.2f}) + Fermi C-H ~2720 ({ch_ald:.2f})"))

    return sorted(results, key=lambda x: -x[0])


# ══════════════════════════════════════════════════════════════════════
# 피크 검출 + FWHM 계산
# ══════════════════════════════════════════════════════════════════════
peak_idx, props = find_peaks(
    sample_vec,
    height=PEAK_HEIGHT,
    prominence=PEAK_PROMINENCE,
    distance=PEAK_DISTANCE,
)
peak_wn  = GRID[peak_idx]
peak_val = sample_vec[peak_idx]

# FWHM 계산 (cm⁻¹ 단위)
widths_pts, _, _, _ = peak_widths(sample_vec, peak_idx, rel_height=0.5)
wn_step = (WN_MAX - WN_MIN) / (N_GRID - 1)
peak_fwhm = widths_pts * wn_step

# ── 개별 피크 테이블 출력 ─────────────────────────────────────────────
rows = []
for wn, val, fwhm in sorted(zip(peak_wn, peak_val, peak_fwhm), key=lambda x: -x[1]):
    name, _, note = assign_group(wn)
    rows.append({
        "Wavenumber (cm⁻¹)": f"{wn:.1f}",
        "Intensity":          f"{val:.3f}",
        "FWHM (cm⁻¹)":       f"{fwhm:.1f}",
        "Assigned Group":     name,
        "Note":               note,
    })
df_peaks = pd.DataFrame(rows)
print(f"검출된 피크: {len(peak_idx)}개\n")
display(df_peaks)

# ── 복수 피크 패턴 분석 ───────────────────────────────────────────────
print("\n" + "═"*65)
print("  복수 피크 패턴 분석 (Functional Group Pattern Matching)")
print("═"*65)
patterns = detect_patterns(peak_wn, peak_val)
if patterns:
    for conf, pname, evidence in patterns:
        bar = "█" * int(conf * 20)
        print(f"  {conf*100:5.1f}% {bar:<20}  {pname}")
        print(f"         └ {evidence}")
else:
    print("  패턴 없음 — 파라미터 조정 필요")
print("═"*65)

작용기 규칙 로드: 47개  ←  data/func_groups.csv
검출된 피크: 3개



,Wavenumber (cm⁻¹),Intensity,FWHM (cm⁻¹),Assigned Group,Note
0,661.5,1.000,106.5,unknown,
1,3992.3,0.833,76.8,unknown,
2,3668.6,0.334,111.5,O-H stretch (free -OH),날카로운 자유 수산기



═════════════════════════════════════════════════════════════════
  복수 피크 패턴 분석 (Functional Group Pattern Matching)
═════════════════════════════════════════════════════════════════
  패턴 없음 — 파라미터 조정 필요
═════════════════════════════════════════════════════════════════


## 3-3. 피크 주석이 있는 스펙트럼 그래프

In [12]:
fig_peak = go.Figure()

# ── 스펙트럼 선 ───────────────────────────────────────────────────────
fig_peak.add_trace(go.Scatter(
    x=GRID,
    y=sample_vec,
    mode="lines",
    name=SAMPLE_LABEL,
    line=dict(color="#374151", width=1.8),
    hovertemplate="%{x:.1f} cm⁻¹ | Abs: %{y:.4f}<extra></extra>",
))

# ── 피크 마커 (작용기별 색상) ─────────────────────────────────────────
for wn, val, fwhm in zip(peak_wn, peak_val, peak_fwhm):
    group_name, color, note = assign_group(wn)
    fig_peak.add_trace(go.Scatter(
        x=[wn],
        y=[val],
        mode="markers",
        marker=dict(color=color, size=9, symbol="circle",
                    line=dict(color="white", width=1.5)),
        name=group_name,
        legendgroup=group_name,
        showlegend=not any(t.name == group_name for t in fig_peak.data[:-1]),
        hovertemplate=(
            f"<b>{wn:.1f} cm⁻¹</b><br>"
            f"{group_name}<br>"
            f"Intensity: {val:.4f}<br>"
            f"FWHM: {fwhm:.1f} cm⁻¹<br>"
            f"<i>{note}</i>"
            "<extra></extra>"
        ),
    ))

# ── 피크 수직선 + 파수 라벨 (상위 25개, 짝홀 높이 교차) ─────────────
top_peaks = sorted(zip(peak_wn, peak_val, peak_fwhm),
                   key=lambda x: -x[1])[:25]
annotations = []
for i, (wn, val, fwhm) in enumerate(top_peaks):
    _, color, _ = assign_group(wn)
    y_label = val + 0.07 + (0.07 if i % 2 == 0 else 0.0)
    annotations.append(dict(
        x=wn, y=y_label,
        text=f"<b>{wn:.0f}</b>",
        showarrow=True,
        arrowhead=0, arrowcolor=color, arrowwidth=1,
        ax=0, ay=-28,
        font=dict(size=9, color=color),
        bgcolor="rgba(255,255,255,0.8)",
        borderpad=1,
    ))
    fig_peak.add_shape(
        type="line", x0=wn, x1=wn, y0=0, y1=val,
        line=dict(color=color, width=0.8, dash="dot"),
    )

fig_peak.update_layout(
    title=dict(text=f"FTIR Peak Analysis — {SAMPLE_LABEL}", font=dict(size=18), x=0.01),
    xaxis=dict(
        title="Wavenumber (cm⁻¹)",
        range=[WN_MAX, WN_MIN],
        showgrid=True, gridcolor="#e8e8e8",
        tickmode="linear", dtick=500,
        minor=dict(showgrid=True, gridcolor="#f4f4f4"),
    ),
    yaxis=dict(
        title="Normalized Absorbance",
        showgrid=True, gridcolor="#e8e8e8",
        range=[-0.05, max(peak_val) * 1.6 if len(peak_val) else 1.3],
    ),
    annotations=annotations,
    legend=dict(
        title="<b>작용기 (범례 클릭)</b>",
        itemclick="toggle",
        itemdoubleclick="toggleothers",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#ccc", borderwidth=1,
        font=dict(size=11),
    ),
    plot_bgcolor="white",
    paper_bgcolor="#fafafa",
    height=620,
    hovermode="closest",
    margin=dict(l=70, r=30, t=70, b=60),
)
fig_peak.show()
print(f"\n피크 {len(peak_idx)}개 표시 (상위 {len(top_peaks)}개 라벨)")


피크 3개 표시 (상위 3개 라벨)


## 4. 라이브러리 로드 및 매칭 행렬 구성
217개 라이브러리 스펙트럼을 동일하게 전처리하고 코사인 유사도를 계산합니다.

In [13]:
manifest = pd.read_csv(MANIFEST)
manifest["category"] = manifest["file"].apply(lambda x: x.split("/")[0])

lib_vecs = []
valid_idx = []

for i, row in manifest.iterrows():
    fpath = os.path.join(LIBRARY_DIR, row["file"])
    try:
        df_lib = load_csv(fpath)
        if len(df_lib) < 10:
            continue
        vec = preprocess(df_lib["wn"].values, df_lib["y"].values)
        lib_vecs.append(vec)
        valid_idx.append(i)
    except Exception:
        pass

lib_matrix = np.array(lib_vecs)          # shape: (N_lib, N_GRID)
valid_meta = manifest.loc[valid_idx].reset_index(drop=True)

print(f"라이브러리 로드: {len(lib_vecs)}개 스펙트럼")

라이브러리 로드: 217개 스펙트럼


## 5. 코사인 유사도 계산 및 상위 결과 출력

In [14]:
scores = cosine_similarity(sample_vec.reshape(1, -1), lib_matrix)[0]

valid_meta = valid_meta.copy()
valid_meta["cosine_similarity"] = scores
valid_meta["match_pct"] = (scores * 100).round(2)

# 소재별 최고 점수만 남기기 (중복 측정 중 best pick)
best_per_material = (
    valid_meta.sort_values("cosine_similarity", ascending=False)
    .drop_duplicates(subset="material")
    .head(TOP_N)
    .reset_index(drop=True)
)

# 카테고리 라벨 매핑
CAT_LABEL = {
    "01_battery":             "Battery",
    "02_steel_coating":       "Steel Coating",
    "03_engineering_plastic": "Engineering Plastic",
    "04_elastomers_seals":    "Elastomers & Seals",
    "05_ceramic_inorganic":   "Ceramic / Inorganic",
}
best_per_material["category_label"] = best_per_material["category"].map(CAT_LABEL)

display_cols = ["material", "category_label", "source", "match_pct", "file"]
print(f"{'Rank':<5} {'Match %':>8}  {'Material':<40} {'Category':<25} {'Source'}")
print("-" * 105)
for rank, row in best_per_material.iterrows():
    print(f"{rank+1:<5} {row['match_pct']:>7.2f}%  {row['material']:<40} "
          f"{row['category_label']:<25} {row['source']}")

Rank   Match %  Material                                 Category                  Source
---------------------------------------------------------------------------------------------------------
1       96.97%  Polyethylene Terephthalate (PET)         Engineering Plastic       Zenodo (FT-IR spectra)
2       93.95%  Triglycerides (TAGs)                     Engineering Plastic       Zenodo (FT-IR spectra)
3       92.77%  Polyhydroxybutyrate-valerate (PHBV)      Engineering Plastic       Zenodo (FT-IR spectra)
4       89.70%  Terephthalic Acid (TPA)                  Engineering Plastic       Zenodo (FT-IR spectra)
5       87.91%  ABS (Acrylonitrile Butadiene Styrene)    Engineering Plastic       figshare (thermoplastic dataset)
6       83.76%  Stearic Acid                             Steel Coating             NIST WebBook
7       83.70%  Propylene Carbonate (PC)                 Battery                   NIST WebBook
8       83.51%  Triethyl Phosphate                       Steel Coating  

## 6. 매칭 점수 바 차트

In [15]:
CAT_COLORS = {
    "Battery":             "#2563eb",
    "Steel Coating":       "#dc2626",
    "Engineering Plastic": "#16a34a",
    "Elastomers & Seals":  "#ea580c",
    "Ceramic / Inorganic": "#7c3aed",
}

fig_bar = go.Figure(go.Bar(
    x=best_per_material["match_pct"],
    y=best_per_material["material"],
    orientation="h",
    marker_color=[CAT_COLORS.get(c, "#888") for c in best_per_material["category_label"]],
    text=best_per_material["match_pct"].apply(lambda v: f"{v:.2f}%"),
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Match: %{x:.2f}%<br>"
        "<extra></extra>"
    ),
))

fig_bar.update_layout(
    title=f"Top {TOP_N} Spectral Matches — {SAMPLE_LABEL}",
    xaxis=dict(title="Cosine Similarity (%)", range=[0, 110]),
    yaxis=dict(autorange="reversed"),
    height=max(350, TOP_N * 40),
    plot_bgcolor="white", paper_bgcolor="#fafafa",
    margin=dict(l=200, r=60, t=60, b=50),
)
fig_bar.show()

## 7. 미지 시료 vs 상위 매칭 스펙트럼 비교 플롯
미지 시료(검정)와 상위 N개 매칭 스펙트럼을 offset을 주어 겹쳐 비교합니다.

In [18]:
top_matches = best_per_material.head(PLOT_TOP_N)
OFFSET_STEP = 1.3   # 스펙트럼 간 세로 간격

fig_cmp = go.Figure()
cmp_annotations = []

# ── 피크 마커/라벨 추가 헬퍼 ─────────────────────────────────────────
def add_peak_markers(fig, vec, offset, color, label_prefix=""):
    """vec에서 피크를 검출하여 마커와 파수 라벨을 추가한다."""
    p_idx, _ = find_peaks(vec, height=PEAK_HEIGHT, prominence=PEAK_PROMINENCE,
                          distance=PEAK_DISTANCE)
    if len(p_idx) == 0:
        return
    p_wn  = GRID[p_idx]
    p_val = vec[p_idx] + offset  # offset 적용

    # 마커 trace
    fig.add_trace(go.Scatter(
        x=p_wn,
        y=p_val,
        mode="markers",
        marker=dict(color=color, size=7, symbol="circle-open",
                    line=dict(color=color, width=2)),
        name=f"{label_prefix}peaks",
        legendgroup=f"{label_prefix}peaks",
        showlegend=False,
        hovertemplate="%{x:.1f} cm⁻¹<br>%{y:.3f}<extra></extra>",
    ))

    # 상위 강도 피크만 라벨 (최대 8개)
    top_n = min(8, len(p_idx))
    top_sort = sorted(zip(p_wn, p_val, vec[p_idx]), key=lambda x: -x[2])[:top_n]
    for i, (wn_p, yy, raw_v) in enumerate(top_sort):
        y_ann = yy + 0.06 + (0.05 if i % 2 == 0 else 0.0)
        cmp_annotations.append(dict(
            x=wn_p, y=y_ann,
            text=f"<b>{wn_p:.0f}</b>",
            showarrow=True,
            arrowhead=0, arrowcolor=color, arrowwidth=1,
            ax=0, ay=-22,
            font=dict(size=8, color=color),
            bgcolor="rgba(255,255,255,0.75)",
            borderpad=1,
        ))


# ── 미지 시료 (맨 위) ────────────────────────────────────────────────
total_offset = PLOT_TOP_N * OFFSET_STEP
fig_cmp.add_trace(go.Scatter(
    x=GRID, y=sample_vec + total_offset,
    mode="lines",
    name=f"★ {SAMPLE_LABEL}",
    line=dict(color="black", width=2),
    hovertemplate=f"<b>{SAMPLE_LABEL}</b><br>%{{x:.1f}} cm⁻¹<extra></extra>",
))
add_peak_markers(fig_cmp, sample_vec, total_offset, "black", "sample")

# ── 매칭 라이브러리 스펙트럼 ─────────────────────────────────────────
PALETTE = ["#2563eb", "#dc2626", "#16a34a", "#ea580c", "#7c3aed"]
for rank, (_, row) in enumerate(top_matches.iterrows()):
    fpath = os.path.join(LIBRARY_DIR, row["file"])
    try:
        df_lib = load_csv(fpath)
        vec = preprocess(df_lib["wn"].values, df_lib["y"].values)
    except Exception:
        continue

    offset = (PLOT_TOP_N - 1 - rank) * OFFSET_STEP
    color  = PALETTE[rank % len(PALETTE)]
    label  = f"#{rank+1} {row['material']}  ({row['match_pct']:.1f}%)"

    fig_cmp.add_trace(go.Scatter(
        x=GRID, y=vec + offset,
        mode="lines",
        name=label,
        line=dict(color=color, width=1.4),
        hovertemplate=(f"<b>{row['material']}</b><br>%{{x:.1f}} cm⁻¹<br>"
                       f"Match: {row['match_pct']:.1f}%<extra></extra>"),
    ))
    add_peak_markers(fig_cmp, vec, offset, color, label)

fig_cmp.update_layout(
    title=f"Spectral Comparison — {SAMPLE_LABEL} vs Top {PLOT_TOP_N} Matches",
    xaxis=dict(
        title="Wavenumber (cm⁻¹)",
        range=[WN_MAX, WN_MIN],
        showgrid=True, gridcolor="#e8e8e8",
        tickmode="linear", dtick=500,
        minor=dict(showgrid=True, gridcolor="#f4f4f4"),
    ),
    yaxis=dict(
        title="Normalized Absorbance (offset)",
        showgrid=False, zeroline=False,
        showticklabels=False,
    ),
    annotations=cmp_annotations,
    legend=dict(
        title="<b>범례 클릭으로 표시/숨기기</b>",
        itemclick="toggle",
        itemdoubleclick="toggleothers",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#ccc", borderwidth=1,
    ),
    plot_bgcolor="white", paper_bgcolor="#fafafa",
    height=700,
    hovermode="closest",
    margin=dict(l=60, r=20, t=70, b=60),
)
fig_cmp.show()

## 8. 결과 저장 (선택)

In [19]:
stem = os.path.splitext(os.path.basename(SAMPLE_FILE))[0]

# 매칭 결과 CSV
csv_out = f"{stem}_matches.csv"
best_per_material[display_cols].to_csv(csv_out, index=False, encoding="utf-8-sig")
print(f"매칭 결과 저장: {csv_out}")

# 비교 그래프 HTML
html_out = f"{stem}_comparison.html"
fig_cmp.write_html(html_out, include_plotlyjs="cdn",
                   config={"scrollZoom": True,
                           "toImageButtonOptions": {"format": "svg", "scale": 2}})
print(f"비교 그래프 저장: {html_out}")

매칭 결과 저장: sample_TR(transmittance).0_matches.csv
비교 그래프 저장: sample_TR(transmittance).0_comparison.html
